In [ ]:
# =========================================================
# 1. CONFIGURATION
# =========================================================

# Client IDs for Frost API
import os
FROST_CLIENT_ID = os.environ.get("FROST_CLIENT_ID", "")
FROST_CLIENT_SECRET = os.environ.get("FROST_CLIENT_SECRET", "")
if not FROST_CLIENT_ID:
    raise RuntimeError("Set FROST_CLIENT_ID as an environment variable before running this notebook")
# Date range for data retrieval
DATE_START = "2014-01-01"
DATE_END   = "2024-12-31"

# Selected weather stations for Oslo and Akershus regional approximation
STATION_IDS = [
    "SN18700",  # Oslo Blindern
    "SN4780",   # Gardermoen
    "SN17850"   # Lillestrøm
]

# Weather elements to download
WEATHER_ELEMENTS = [
    "mean(air_temperature P1D)",
    "sum(precipitation_amount P1D)"
]


# =========================================================
# 2. IMPORTS
# =========================================================
import requests
import base64
import pandas as pd
import numpy as np
from google.colab import files
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
import seaborn as sns


# =========================================================
# 3. WEATHER DATA DOWNLOAD (DAILY)
# =========================================================
print("Downloading daily weather data from MET Norway Frost API...")

auth = base64.b64encode(f"{FROST_CLIENT_ID}:{FROST_CLIENT_SECRET}".encode()).decode()
headers = {"Authorization": f"Basic {auth}", "User-Agent": "UiO-Student"}

all_weather_raw = []

for station in STATION_IDS:
    print(f"Fetching daily data for station: {station}")
    params = {
        "sources": station,
        "elements": ",".join(WEATHER_ELEMENTS),
        "referencetime": f"{DATE_START}/{DATE_END}"
    }

    r = requests.get(
        "https://frost.met.no/observations/v0.jsonld",
        headers=headers,
        params=params,
        timeout=60
    )

    if r.status_code != 200:
        print(f"Error fetching data for station {station}. HTTP {r.status_code}. Response: {r.text}")
        continue

    try:
        data = r.json()["data"]
    except KeyError:
        print(f"'data' key missing in response for station {station}. Response: {r.text}")
        continue

    for item in data:
        date = pd.to_datetime(item["referenceTime"]).normalize()
        row = {"station": station, "date": date}
        for obs in item["observations"]:
            row[obs["elementId"]] = obs["value"]
        all_weather_raw.append(row)

df_weather_daily = pd.DataFrame(all_weather_raw)

if df_weather_daily.empty:
    raise ValueError("No weather data was downloaded. Check Frost credentials, station IDs, or date range.")

print(f"Raw daily weather rows from all stations: {len(df_weather_daily)}")
print(df_weather_daily.head())

print("\nMissing values in raw weather data:")
print(df_weather_daily.isna().sum())


# =========================================================
# 4. WEATHER DATA AGGREGATION
# =========================================================
# Weather is approximated from nearby stations.
# First aggregate across stations to one regional daily value,
# then aggregate daily regional values to monthly level.

df_weather_daily_region = (
    df_weather_daily
    .groupby("date", as_index=False)
    .agg({
        "mean(air_temperature P1D)": "mean",
        "sum(precipitation_amount P1D)": "mean"
    })
    .rename(columns={
        "mean(air_temperature P1D)": "daily_mean_temperature",
        "sum(precipitation_amount P1D)": "daily_mean_precipitation"
    })
)

df_weather_daily_region["year"] = df_weather_daily_region["date"].dt.year
df_weather_daily_region["month"] = df_weather_daily_region["date"].dt.month

df_weather_monthly = (
    df_weather_daily_region
    .groupby(["year", "month"], as_index=False)
    .agg(
        mean_air_temperature=("daily_mean_temperature", "mean"),
        total_precipitation=("daily_mean_precipitation", "sum")
    )
)

print(f"\nAggregated monthly weather rows: {len(df_weather_monthly)}")
print(df_weather_monthly.head())


# =========================================================
# 5. ACCIDENT DATA LOADING AND CLEANING (MONTHLY)
# =========================================================
print("\nUPLOAD TRINE CSV FILE (monthly accident data)")
uploaded = files.upload()
acc_file = list(uploaded.keys())[0]

acc_monthly = pd.read_csv(acc_file, sep=';', encoding='utf-8-sig')

acc_monthly.columns = [col.strip().replace('\n', '') for col in acc_monthly.columns]

print("\nOriginal accident columns:")
print(acc_monthly.columns.tolist())

acc_monthly = acc_monthly.rename(columns={
    'År': 'year',
    'Måned': 'month_name',
    'Dødsulykke': 'fatal_accidents',
    'Ulykke med hardt skadde': 'serious_injury_accidents',
    'Ulykke med lettere skadde': 'minor_injury_accidents',
})

month_mapping = {
    'Januar': 1, 'Februar': 2, 'Mars': 3, 'April': 4, 'Mai': 5, 'Juni': 6,
    'Juli': 7, 'August': 8, 'September': 9, 'Oktober': 10, 'November': 11, 'Desember': 12
}

acc_monthly['month'] = acc_monthly['month_name'].map(month_mapping)

if acc_monthly['month'].isna().any():
    print("\nWarning: Unmapped month names found:")
    print(acc_monthly.loc[acc_monthly['month'].isna(), ['year', 'month_name']].drop_duplicates())

study_start_year = pd.to_datetime(DATE_START).year
study_end_year = pd.to_datetime(DATE_END).year

# Restrict accident data to the study period so it matches the weather data.
acc_monthly = acc_monthly[
    (acc_monthly['year'] >= study_start_year) & (acc_monthly['year'] <= study_end_year)
].copy()

# Create a new variable that clearly includes only accidents with personal injury.
acc_monthly['injury_accidents'] = (
    acc_monthly['fatal_accidents'].fillna(0)
    + acc_monthly['serious_injury_accidents'].fillna(0)
    + acc_monthly['minor_injury_accidents'].fillna(0)
)

acc_monthly = acc_monthly[
    ['year', 'month', 'fatal_accidents', 'serious_injury_accidents', 'minor_injury_accidents', 'injury_accidents']
].copy()

duplicate_count = acc_monthly.duplicated(subset=['year', 'month']).sum()
print(f"\nDuplicate year-month rows in accident data: {duplicate_count}")

print("\nCleaned Monthly Accident DataFrame head:")
print(acc_monthly.head())
print(f"Monthly accident data rows: {len(acc_monthly)}")


# =========================================================
# 6. MERGE DATASETS
# =========================================================
# Accident data is monthly aggregated.
# Findings should be interpreted as monthly regional associations, not causal effects.

df_weather_monthly['year'] = df_weather_monthly['year'].astype(int)
df_weather_monthly['month'] = df_weather_monthly['month'].astype(int)
acc_monthly['year'] = acc_monthly['year'].astype(int)
acc_monthly['month'] = acc_monthly['month'].astype(int)

df = pd.merge(acc_monthly, df_weather_monthly, on=['year', 'month'], how='inner')

accident_cols = ['fatal_accidents', 'serious_injury_accidents', 'minor_injury_accidents', 'injury_accidents']
df[accident_cols] = df[accident_cols].fillna(0)

df['plotting_date'] = pd.to_datetime(
    df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-01'
)
df = df.sort_values('plotting_date').reset_index(drop=True)

print(f"\nFinal merged dataset rows (one row per month): {len(df)}")
print(df.head())


# =========================================================
# 7. FEATURE ENGINEERING
# =========================================================
conditions = [
    df['month'].isin([12, 1, 2]),
    df['month'].isin([3, 4, 5]),
    df['month'].isin([6, 7, 8]),
    df['month'].isin([9, 10, 11])
]
choices = ['Winter', 'Spring', 'Summer', 'Autumn']
df['season'] = np.select(conditions, choices, default=None) # Changed default from np.nan to None

df['monthly_injury_accident_occurrence'] = (df['injury_accidents'] > 0).astype(int)


# =========================================================
# 8. DATA QUALITY CHECKS
# =========================================================
print("\nMissing values after feature engineering:")
print(df.isna().sum())

print("\nFinal dataset info:")
print(df.info())


# =========================================================
# 9. VISUALIZATION
# =========================================================

# 1. Monthly accident time series
plt.figure(figsize=(12, 6))
sns.lineplot(x='plotting_date', y='injury_accidents', data=df, marker='o')
plt.xlabel('Date')
plt.ylabel('Monthly Personal-Injury Accident Counts')
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Monthly mean temperature time series
plt.figure(figsize=(12, 6))
sns.lineplot(x='plotting_date', y='mean_air_temperature', data=df, marker='o')
plt.xlabel('Date')
plt.ylabel('Monthly Mean Air Temperature (°C)')
plt.grid(True)
plt.tight_layout()
plt.show()

# 3. Monthly precipitation time series
plt.figure(figsize=(12, 6))
sns.lineplot(x='plotting_date', y='total_precipitation', data=df, marker='o')
plt.xlabel('Date')
plt.ylabel('Monthly Total Precipitation (mm)')
plt.grid(True)
plt.tight_layout()
plt.show()

# 4. Scatter plot: temperature vs accidents
plt.figure(figsize=(8, 6))
sns.regplot(
    x='mean_air_temperature',
    y='injury_accidents',
    data=df,
    scatter_kws={'alpha': 0.6}
)
plt.xlabel('Monthly Mean Air Temperature (°C)')
plt.ylabel('Monthly Personal-Injury Accident Counts')
plt.grid(True)
plt.tight_layout()
plt.show()

# 5. Scatter plot: precipitation vs accidents
plt.figure(figsize=(8, 6))
sns.regplot(
    x='total_precipitation',
    y='injury_accidents',
    data=df,
    scatter_kws={'alpha': 0.6}
)
plt.xlabel('Monthly Total Precipitation (mm)')
plt.ylabel('Monthly Personal-Injury Accident Counts')
plt.grid(True)
plt.tight_layout()
plt.show()

# 6. Accident seasonality by month
plt.figure(figsize=(10, 6))
sns.boxplot(x='month', y='injury_accidents', data=df)
plt.xlabel('Month')
plt.ylabel('Monthly Personal-Injury Accident Counts')
plt.xticks(
    ticks=np.arange(0, 12),
    labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 7. Heatmap: year vs month accidents
pivot_accidents = df.pivot_table(index='year', columns='month', values='injury_accidents', aggfunc='sum')

plt.figure(figsize=(12, 8))
sns.heatmap(
    pivot_accidents,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Monthly Personal-Injury Accident Counts'}
)
plt.xlabel('Month')
plt.ylabel('Year')
plt.xticks(
    ticks=np.arange(0.5, 12.5, 1),
    labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    rotation=0
)
plt.tight_layout()
plt.show()

# 8. Correlation matrix
correlation_df = df[[
    'injury_accidents',
    'fatal_accidents',
    'serious_injury_accidents',
    'minor_injury_accidents',
    'mean_air_temperature',
    'total_precipitation',
    'month',
    'year'
]]

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.tight_layout()
plt.show()

# 9. Severity trends over time
plt.figure(figsize=(12, 7))
sns.lineplot(x='plotting_date', y='fatal_accidents', data=df, label='Fatal Accidents', marker='o')
sns.lineplot(x='plotting_date', y='serious_injury_accidents', data=df, label='Accidents with Serious Injuries', marker='o')
sns.lineplot(x='plotting_date', y='minor_injury_accidents', data=df, label='Accidents with Minor Injuries', marker='o')
plt.xlabel('Date')
plt.ylabel('Number of Accidents')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



# =========================================================
# 11. SAVING FINAL DATASET
# =========================================================


variables_for_stats = [
    'injury_accidents',
    'fatal_accidents',
    'serious_injury_accidents',
    'minor_injury_accidents',
    'mean_air_temperature',
    'total_precipitation'
]

descriptive_stats = df[variables_for_stats].agg(['mean', 'std', 'min', 'max']).T
descriptive_stats.columns = ['Mean', 'Std. Dev.', 'Min', 'Max']

# Map column names to more readable labels for the table
descriptive_stats_display = descriptive_stats.rename(index={
    'injury_accidents': 'Injury accidents',
    'fatal_accidents': 'Fatal accidents',
    'serious_injury_accidents': 'Serious-injury accidents',
    'minor_injury_accidents': 'Minor-injury accidents',
    'mean_air_temperature': 'Mean air temperature',
    'total_precipitation': 'Total precipitation'
})

# Format to 2 decimal places
descriptive_stats_display = descriptive_stats_display.round(2)

print("Descriptive statistics for the main variables:")
print(descriptive_stats_display.to_latex(
    caption='Descriptive statistics for the main variables',
    label='tab:descriptive_stats',
    header=True,
    float_format="%.2f",
    column_format='lrrrr'
))

# 1. Check for required columns
required_columns = [
    'year', 'month', 'injury_accidents', 'mean_air_temperature', 'total_precipitation'
]

missing_cols = [col for col in required_columns if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in DataFrame: {', '.join(missing_cols)}")
else:
    print("All required columns are present.")

# Ensure year and month are integers for date creation
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)

# 2. Create a date column if it does not already exist
if 'date' not in df.columns:
    df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-01')
    print("Created 'date' column.")
else:
    print("'date' column already exists.")

# 3. Sort the data chronologically
df = df.sort_values(by='date').reset_index(drop=True)
print("Data sorted chronologically.")

# 4. Create improved time variables
min_year = df['year'].min()
df['year_index'] = df['year'] - min_year
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print("Created year_index, month_sin, and month_cos variables.")

# 5. Define predictors and target variable
predictors = [
    'mean_air_temperature',
    'total_precipitation',
    'month_sin',
    'month_cos',
    'year_index'
]
target = 'injury_accidents'

X = df[predictors]
y = df[target]

# Add constant for statsmodels GLM models
X_sm = sm.add_constant(X, prepend=True)

print("Predictors and target variables defined.")
print(f"Predictors: {predictors}")
print(f"Target: {target}")

# 7. Split the data chronologically into 80% training and 20% holdout/test data
split_point = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

X_sm_train, X_sm_test = X_sm.iloc[:split_point], X_sm.iloc[split_point:]

print(f"Data split: {len(X_train)} training samples, {len(X_test)} holdout samples.")

# 8. Fit a Poisson regression model
print("\nFitting Poisson Regression Model...")
poisson_model = sm.GLM(y_train, X_sm_train, family=sm.families.Poisson())
poisson_results = poisson_model.fit()
print("Poisson model fitted.")

import os
os.makedirs("figures", exist_ok=True)

coef_poisson_updated = poisson_results.params.drop("const", errors="ignore")

import os
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

coef_poisson_updated = poisson_results.params.drop("const", errors="ignore")
pvals = poisson_results.pvalues.drop("const", errors="ignore")

pretty_names = {
    "mean_air_temperature": "Mean air temperature",
    "total_precipitation": "Total precipitation",
    "month_sin": "Month sine",
    "month_cos": "Month cosine",
    "year_index": "Year index"
}

coef_poisson_updated = coef_poisson_updated.rename(index=pretty_names)
pvals = pvals.rename(index=pretty_names)

plot_df = (
    pd.DataFrame({
        "Coefficient": coef_poisson_updated,
        "p-value": pvals
    })
    .sort_values("Coefficient", ascending=True)
)

def sig_marker(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    elif p < 0.10:
        return "†"
    else:
        return ""

fig, ax = plt.subplots(figsize=(9, 5.2))

bars = ax.barh(plot_df.index, plot_df["Coefficient"])

ax.axvline(0, linewidth=1)
ax.set_xlabel("Coefficient Value")
ax.set_ylabel("Feature")
ax.grid(axis="x", linestyle="--", alpha=0.6)
ax.set_axisbelow(True)

xmin = plot_df["Coefficient"].min() - 0.025
xmax = plot_df["Coefficient"].max() + 0.025
ax.set_xlim(xmin, xmax)

for bar, (coef, pval) in zip(bars, zip(plot_df["Coefficient"], plot_df["p-value"])):
    y = bar.get_y() + bar.get_height() / 2
    label = f"{coef:.4f}{sig_marker(pval)}"

    if coef < 0:
        ax.text(coef - 0.003, y, label, va="center", ha="right", fontsize=9)
    else:
        ax.text(coef + 0.003, y, label, va="center", ha="left", fontsize=9)



plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("figures/plot10_poisson_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()



# 9. Calculate the Poisson overdispersion statistic
print("\nCalculating Poisson overdispersion statistic...")
poisson_predictions_train = poisson_results.predict(X_sm_train)
pearson_chi2 = np.sum((y_train - poisson_predictions_train)**2 / poisson_predictions_train)
residual_df = poisson_results.df_resid
overdispersion_statistic = pearson_chi2 / residual_df

print(f"Pearson Chi-square: {pearson_chi2:.2f}")
print(f"Residual Degrees of Freedom: {residual_df:.0f}")
print(f"Overdispersion Statistic (Pearson chi-square / df): {overdispersion_statistic:.2f}")

if overdispersion_statistic > 1.2: # A common heuristic for indicating overdispersion
    print("Result indicates potential overdispersion, a Negative Binomial model may be more appropriate.")
else:
    print("Result does not strongly indicate overdispersion.")

# 10. Fit a Negative Binomial regression model as a robustness check for overdispersion
from statsmodels.discrete.discrete_model import NegativeBinomial

print("\nFitting Negative Binomial Regression Model with estimated dispersion...")
negbin_model = NegativeBinomial(y_train, X_sm_train)
negbin_results = negbin_model.fit(disp=False)
negbin_pred = negbin_results.predict(X_sm_test)

print(negbin_results.summary())





# 11. Fit a RandomForestRegressor as a simple machine-learning benchmark
from sklearn.ensemble import RandomForestRegressor

print("\nFitting RandomForestRegressor Model...")
rf_model = RandomForestRegressor(random_state=42, n_estimators=100) # Added n_estimators for better performance
rf_model.fit(X_train, y_train)
print("RandomForestRegressor model fitted.")


# 12. Evaluate all three models on the holdout set using MAE and RMSE
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("\nEvaluating models on the holdout set...")

# Poisson Model Predictions
poisson_pred = poisson_results.predict(X_sm_test)
poisson_mae = mean_absolute_error(y_test, poisson_pred)
poisson_rmse = np.sqrt(mean_squared_error(y_test, poisson_pred))

# Negative Binomial Model Predictions
negbin_pred = negbin_results.predict(X_sm_test)
negbin_mae = mean_absolute_error(y_test, negbin_pred)
negbin_rmse = np.sqrt(mean_squared_error(y_test, negbin_pred))

# Random Forest Model Predictions
rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

metrics_data = {
    'Model': ['Poisson Regression', 'Negative Binomial Regression', 'Random Forest Regressor'],
    'MAE': [poisson_mae, negbin_mae, rf_mae],
    'RMSE': [poisson_rmse, negbin_rmse, rf_rmse]
}
model_comparison_df = pd.DataFrame(metrics_data)

print("Model evaluation complete.")

# 13. Print a clean model comparison table
print("\nModel Comparison Table:")
print(model_comparison_df.to_string(index=False))


# 14. Print the Poisson model summary
print("\n======================================================")
print("Poisson Regression Model Summary:")
print(poisson_results.summary())
print("======================================================")


# 15. Print a Poisson coefficient table with Variable, Coefficient, IRR, p-value
print("\nPoisson Coefficient Table:")
poisson_coef_df = pd.DataFrame({
    'Variable': poisson_results.params.index,
    'Coefficient': poisson_results.params.values,
    'IRR': np.exp(poisson_results.params.values),
    'p-value': poisson_results.pvalues.values
})
# Format p-values
poisson_coef_df['p-value'] = poisson_coef_df['p-value'].apply(lambda x: f"{x:.3f}" if x >= 0.001 else "<0.001")
print(poisson_coef_df.to_string(index=False))

# 16. Print the negative binomial model summary
print("\n======================================================")
print("Negative Binomial Regression Model Summary:")
print(negbin_results.summary())
print("======================================================")

# 17. Print random forest feature importances
print("\nRandom Forest Feature Importances:")
rf_feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)
print(rf_feature_importances.to_string(index=False))


# 18. Save the following outputs as CSV files
print("\nSaving results to CSV files...")
model_comparison_df.to_csv('model_comparison.csv', index=False)
poisson_coef_df.to_csv('poisson_coefficients_irr.csv', index=False)
rf_feature_importances.to_csv('random_forest_feature_importance.csv', index=False)
print("CSV files saved: model_comparison.csv, poisson_coefficients_irr.csv, random_forest_feature_importance.csv")


# 19. Create and save two figures
import matplotlib.pyplot as plt

print("\nCreating and saving figures...")

# Figure 1: Actual vs. Predicted Holdout
plt.figure(figsize=(14, 7))
plt.plot(df['date'].iloc[split_point:], y_test, label='Actual Accident Counts', marker='o', linestyle='-')
plt.plot(df['date'].iloc[split_point:], poisson_pred, label='Poisson Predictions', marker='x', linestyle='--')
plt.plot(df['date'].iloc[split_point:], negbin_pred, label='Negative Binomial Predictions', marker='s', linestyle='-.')
plt.plot(df['date'].iloc[split_point:], rf_pred, label='Random Forest Predictions', marker='^', linestyle=':')

plt.xlabel('Date')
plt.ylabel('Monthly Personal-Injury Accident Counts')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('actual_vs_predicted_holdout.png')
plt.show()


print("\nSaving final dataset...")
df.to_csv("final_dataset.csv", index=False)
files.download("final_dataset.csv")

print("DONE: Full pipeline execution complete.")

# Figure 2: Random Forest Feature Importance
import os
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

# Sort so the most important feature appears at the top
rf_plot = rf_feature_importances.sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.2))

bars = ax.barh(rf_plot["Feature"], rf_plot["Importance"])

ax.set_xlabel("Feature importance")
ax.set_ylabel("Feature")
ax.grid(axis="x", linestyle="--", alpha=0.6)
ax.set_axisbelow(True)

# Add space on the right for value labels
xmax = rf_plot["Importance"].max() + 0.05
ax.set_xlim(0, xmax)

# Add numeric labels to bars
offset = 0.005
for bar, value in zip(bars, rf_plot["Importance"]):
    y = bar.get_y() + bar.get_height() / 2
    ax.text(value + offset, y, f"{value:.3f}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.savefig("figures/random_forest_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved: figures/random_forest_feature_importance.png")
print("Figures saved: actual_vs_predicted_holdout.png, random_forest_feature_importance.png")



In [ ]:
from scipy.stats import pearsonr
import pandas as pd # Ensure pandas is imported

if 'df' not in globals():
    print("Error: DataFrame 'df' not found. Please run the preceding cells to load and prepare the data.")
else:
    # Calculate Pearson correlation and p-value for temperature
    corr_temp, p_temp = pearsonr(df['injury_accidents'], df['mean_air_temperature'])

    # Calculate Pearson correlation and p-value for precipitation
    corr_precip, p_precip = pearsonr(df['injury_accidents'], df['total_precipitation'])

    # Create a DataFrame for the table
    bivariate_correlations = pd.DataFrame({
        'Weather variable': ['Mean air temperature', 'Total precipitation'],
        'Pearson correlation': [corr_temp, corr_precip],
        'p-value': [p_temp, p_precip],
        'Interpretation': ['Positive association', 'Weak positive association']
    })

    # Format the correlation and p-value columns
    bivariate_correlations['Pearson correlation'] = bivariate_correlations['Pearson correlation'].map('{:.2f}'.format)
    bivariate_correlations['p-value'] = bivariate_correlations['p-value'].map('{:.3f}'.format)

    # Generate the LaTeX table
    latex_table = bivariate_correlations.to_latex(
        index=False,
        caption='Bivariate correlations between weather variables and monthly personal-injury accident counts',
        label='tab:bivariate_correlations',
        column_format='lrrl',
        header=True
    )

    # Add LaTeX table environment manually for [H] placement and
    # bold column names as requested in the template.
    latex_table_final = """
\\begin{{table}}[H]
\\centering
\\small
{} \n{}
\\end{{table}}
""".format(
        latex_table.replace('Pearson correlation', '\\textbf{Pearson correlation}')
                   .replace('p-value', '\\textbf{p-value}')
                   .replace('Interpretation', '\\textbf{Interpretation}')
                   .replace('Weather variable', '\\textbf{Weather variable}'),
        '\\caption{{Bivariate correlations between weather variables and monthly personal-injury accident counts}}\\label{{tab:bivariate_correlations}}'
    )

    print(latex_table_final)

In [ ]:
from scipy.stats import pearsonr
import pandas as pd

# Variables to correlate with monthly personal-injury accident counts
correlation_vars = {
    "Mean air temperature": "mean_air_temperature",
    "Total precipitation": "total_precipitation"
}

target = "injury_accidents"

results = []

for label, var in correlation_vars.items():
    # Drop missing values only for the two variables being compared
    temp_df = df[[target, var]].dropna()

    r, p = pearsonr(temp_df[var], temp_df[target])

    if abs(r) >= 0.5:
        interpretation = "Moderate positive association" if r > 0 else "Moderate negative association"
    elif abs(r) >= 0.3:
        interpretation = "Weak to moderate positive association" if r > 0 else "Weak to moderate negative association"
    elif abs(r) > 0:
        interpretation = "Weak positive association" if r > 0 else "Weak negative association"
    else:
        interpretation = "No linear association"

    results.append({
        "Weather variable": label,
        "Pearson correlation": r,
        "p-value": p,
        "Interpretation": interpretation
    })

correlation_table = pd.DataFrame(results)

# Display rounded version
correlation_table["Pearson correlation"] = correlation_table["Pearson correlation"].round(2)
correlation_table["p-value"] = correlation_table["p-value"].apply(
    lambda x: "<0.001" if x < 0.001 else round(x, 3)
)

print(correlation_table)